# 🎨 Neural Canvas: Interactive Style Transfer Demo

This notebook demonstrates the **Neural Canvas** PyTorch library:
- Hardware detection & VGG19 feature extraction
- Stylization with curated public-domain masterwork presets
- Custom style image path uploads
- Progressive optimization generator (`engine.run()`) with step previews
- Resolution-invariant TV regularization (`tv_weight = 0.05`)

### 1. Import Packages & Hardware Verification

In [1]:
import sys
from pathlib import Path
from PIL import Image

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from neural_canvas import (
    StyleTransferConfig,
    StyleTransferEngine,
    get_device_info,
    resolve_device,
    list_presets,
    get_preset,
    tensor_to_pil,
)

device = resolve_device("auto")
info = get_device_info(device)
print(f"Device: {info['name']} | CUDA Available: {info['is_cuda']}")
if info["is_cuda"]:
    print(f"GPU: {info['gpu_name']} ({info['vram_gb']} GB VRAM, sm_{info['capability']})")

Device: cuda | CUDA Available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU (7.96 GB VRAM, sm_12.0)


### 2. Explore Available Style Presets

In [2]:
presets = list_presets()
print(f"Loaded {len(presets)} curated masterwork style presets:")
for p in presets:
    rw = p.recommended_weights
    print(f"- {p.id:35} : {p.name:32} | cw={rw.get('content_weight'):4.0f}, tv={rw.get('tv_weight'):.2f}")

Loaded 10 curated masterwork style presets:
- a_sunday_afternoon_on_the_island_of_la_grande_jatte : A Sunday Afternoon on the Island of La Grande Jatte | cw= 100, tv=0.05
- composition_vii_1913                : Composition VII                  | cw=  90, tv=0.05
- impression_sunrise                  : Impression, Sunrise              | cw= 120, tv=0.05
- sudden_shower_over_shin_ōhashi_bridge_and_atake : Sudden Shower over Shin-Ōhashi Bridge and Atake | cw=  80, tv=0.05
- the_fighting_temeraire              : The Fighting Temeraire           | cw= 120, tv=0.05
- the_great_wave_off_kanagawa         : The Great Wave off Kanagawa      | cw=  80, tv=0.05
- the_kiss                            : The Kiss                         | cw= 140, tv=0.05
- the_scream                          : The Scream                       | cw=  90, tv=0.05
- the_starry_night                    : The Starry Night                 | cw=  90, tv=0.05
- wheatfield_with_cypresses           : Wheatfield with Cypresses 

### 3. Preset Stylization: Van Gogh's *The Starry Night*

We use the shipped recommended weights for *The Starry Night* (`content_weight = 90.0`, `tv_weight = 0.05`).

In [3]:
content_path = project_root / "assets" / "samples" / "dog.jpg"
starry_preset = get_preset("the_starry_night")
rec = starry_preset.recommended_weights

config = StyleTransferConfig(
    steps=100,
    max_dim=384,
    content_weight=rec["content_weight"],
    style_weight=rec["style_weight"],
    tv_weight=rec["tv_weight"],
    lr_schedule="cosine",
    learning_rate=0.03,
    preview_every=25,
    seed=42,
)

print("Engine configuration:")
print(f"  Resolution: {config.max_dim}px | Steps: {config.steps} | LR: {config.learning_rate} (Cosine)")
print(f"  Loss weights: Content={config.content_weight}, Style={config.style_weight:.1e}, TV={config.tv_weight}")

Engine configuration:
  Resolution: 384px | Steps: 100 | LR: 0.03 (Cosine)
  Loss weights: Content=90.0, Style=1.0e+05, TV=0.05


### 4. Generator-Based Optimization with Progressive Step Previews

`engine.run()` is a Python generator that yields intermediate optimization states and metrics, enabling live UI progress bars and step visualization.

In [4]:
engine = StyleTransferEngine(config=config, device=device)
engine.prepare(content=content_path, style=project_root / starry_preset.image_path)

preview_frames = []
step_losses = []

print("Starting optimization...")
for step_result in engine.run():
    step_losses.append((step_result.step, step_result.total_loss))
    preview_frames.append((step_result.step, tensor_to_pil(step_result.image)))
    print(f"  Step {step_result.step:3d}/{config.steps:3d} | Total Loss: {step_result.total_loss:.2e} | Style: {step_result.style_loss:.2e} | Content: {step_result.content_loss:.2e}")

final_preset_image = preview_frames[-1][1]
final_preset_image.save("starry_night_output.png")
print("Preset stylization complete! Saved starry_night_output.png")

Starting optimization...
  Step  25/100 | Total Loss: 1.33e+05 | Style: 1.26e+05 | Content: 1.20e+03
  Step  50/100 | Total Loss: 8.16e+04 | Style: 7.38e+04 | Content: 1.27e+03
  Step  75/100 | Total Loss: 5.79e+04 | Style: 5.03e+04 | Content: 1.30e+03
  Step 100/100 | Total Loss: 5.24e+04 | Style: 4.48e+04 | Content: 1.31e+03
Preset stylization complete! Saved starry_night_output.png


### 5. Progressive Preview Strip

Stitch the generator step preview frames into a progressive contact strip.

In [5]:
strip_thumbs = [im.resize((150, 150)) for _, im in preview_frames]
total_w = sum(t.width for t in strip_thumbs)
max_h = max(t.height for t in strip_thumbs)
progress_strip = Image.new("RGB", (total_w, max_h))
cur_x = 0
for t in strip_thumbs:
    progress_strip.paste(t, (cur_x, 0))
    cur_x += t.width
progress_strip.save("progress_strip.png")
print(f"Saved progressive optimization strip: {progress_strip.size[0]}x{progress_strip.size[1]}px")

Saved progressive optimization strip: 600x150px


### 6. Custom Style Image Path Example

Neural Canvas accepts arbitrary style image paths or PIL images, applying resolution-invariant scaling and Gram optimization.

In [6]:
custom_style_path = project_root / "styles" / "The Great Wave off Kanagawa.jpg"

custom_config = StyleTransferConfig(
    steps=100,
    max_dim=384,
    content_weight=80.0,
    style_weight=1e5,
    tv_weight=0.05,
    lr_schedule="cosine",
    learning_rate=0.03,
    preview_every=50,
    seed=42,
)

custom_engine = StyleTransferEngine(config=custom_config, device=device)
custom_engine.prepare(content=content_path, style=custom_style_path)

print(f"Optimizing with custom style path: {custom_style_path.name}...")
last_step = list(custom_engine.run())[-1]
custom_output_pil = tensor_to_pil(last_step.image)
custom_output_pil.save("great_wave_custom_output.png")
print("Custom style optimization complete! Saved great_wave_custom_output.png")

Optimizing with custom style path: The Great Wave off Kanagawa.jpg...
Custom style optimization complete! Saved great_wave_custom_output.png


### 7. Triple Comparison: Content, Style & Generated Canvas

In [7]:
with Image.open(content_path) as c_im, Image.open(custom_style_path) as s_im:
    target_size = (300, 300)
    t_content = c_im.convert("RGB").resize(target_size)
    t_style = s_im.convert("RGB").resize(target_size)
    t_result = custom_output_pil.resize(target_size)
    
    comparison_canvas = Image.new("RGB", (900, 300))
    comparison_canvas.paste(t_content, (0, 0))
    comparison_canvas.paste(t_style, (300, 0))
    comparison_canvas.paste(t_result, (600, 0))
    comparison_canvas.save("custom_comparison.png")
    print("Saved side-by-side comparison: custom_comparison.png (Content | Style | Output)")

Saved side-by-side comparison: custom_comparison.png (Content | Style | Output)
